# Minnie CAVE synapse download

Download outgoing synapses for axon-proofread roots in `minnie65_phase3_v1`, materialization **1822**, using the public Delta Lake export. Postsynaptic roots are not restricted.

Requires `caveclient`, `polars`, and `deltalake` (already declared in this repository), plus locally configured CAVE credentials for the proofreading query. Both Delta readers use anonymous GCS access. Availability of the v1822 export must be confirmed when running.

The full download may be large. Set `root_id_limit = 10` for a trial before downloading the full selection. Output files use the absolute directory `/workspaces/ConnectsCommonConnectivity/data/EM v1dd 1196`, independent of the notebook location or kernel working directory.

In [1]:
import bisect
import time
from pathlib import Path

import polars as pl
from caveclient import CAVEclient
from deltalake import DeltaTable

In [8]:
CAVEclient.setup_token("https://global.daf-apis.com") # for minnie
# CAVEclient.setup_token("https://global.em.brain.allentech.org") # for v1dd

Visit https://global.daf-apis.com/sticky_auth/settings/tokens and copy an existing token or create a new one.
All datastacks are configured to use this server address.
You will not need to specify a server address when initializing a client for configured datastacks in the future.
Setup complete!


In [5]:
version = 1822
datastack = "minnie65_phase3_v1"
storage_options = {"GOOGLE_SKIP_SIGNATURE": "true"}
root_id_limit = None

table_path = (
    f"gs://mat_dbs/public/deltalake_exports/{datastack}/"
    f"v{version}/synapses_pni_2/pre_pt_root_id"
)
output_suffix = "" if root_id_limit is None else f"_first_{root_id_limit}_roots"
output_path = (
    Path("/workspaces/ConnectsCommonConnectivity/data/EM v1dd 1196")
    / f"minnie_v{version}_proofread_axon_synapses{output_suffix}.parquet"
)
print(f"Delta export: {table_path}")
print(f"Output: {output_path.resolve()}")

Delta export: gs://mat_dbs/public/deltalake_exports/minnie65_phase3_v1/v1822/synapses_pni_2/pre_pt_root_id
Output: /workspaces/ConnectsCommonConnectivity/data/EM v1dd 1196/minnie_v1822_proofread_axon_synapses.parquet


## Select proofread axons

Use the same materialization version for the proofreading query and synapse export. The optional trial selects a reproducible subset of sorted root IDs.

In [6]:
client = CAVEclient(datastack, version=version)
# status_axon arrives as 't'/'f' strings when the client streams results via CSV, so filter server-side.
proofread_table = client.materialize.query_table(
    "proofreading_status_and_strategy",
    filter_equal_dict={"status_axon": True},
)
proofread_root_ids = sorted(proofread_table["pt_root_id"].dropna().unique().tolist())
print(f"Number of proofread root IDs: {len(proofread_root_ids):,}")

if root_id_limit is not None:
    if not isinstance(root_id_limit, int) or root_id_limit <= 0:
        raise ValueError("root_id_limit must be None or a positive integer.")
    proofread_root_ids = proofread_root_ids[:root_id_limit]
if not proofread_root_ids:
    raise ValueError("No proofread axon root IDs were selected.")
print(f"Root IDs selected for download: {len(proofread_root_ids):,}")


Query was executed using streaming via CSV, which can mangle types. Please upgrade to caveclient > 8.0.0 to avoid type mangling. Because you may have been affected by mangled types, this change is breaking, but it should provide an improved experience.


Number of proofread root IDs: 2,293
Root IDs selected for download: 2,293


## Build the filtered scan

Follow the supplied example's boundary lookup: sorted file maxima are assumed to correspond to consecutive, zero-based `pre_pt_root_id_partition` values in this export. The root-ID filter is applied as well. Drop the same columns as the example, retaining the remaining synapse attributes.

In [7]:
delta_table = DeltaTable(table_path, storage_options=storage_options)
add_actions = pl.from_arrow(delta_table.get_add_actions(flatten=True))
boundaries = add_actions.sort("min.pre_pt_root_id")["max.pre_pt_root_id"].to_list()
proofread_partition_ids = sorted(
    {bisect.bisect_left(boundaries, root_id) for root_id in proofread_root_ids}
)
print(f"Candidate partitions: {len(proofread_partition_ids):,}")

table_plan = pl.scan_delta(table_path, storage_options=storage_options)
table_plan = table_plan.filter(
    pl.col("pre_pt_root_id_partition").is_in(proofread_partition_ids)
    & pl.col("pre_pt_root_id").is_in(proofread_root_ids)
)
table_plan = table_plan.drop(
    "valid",
    "post_pt_position",
    "pre_pt_position",
    "pre_pt_supervoxel_id",
    "post_pt_supervoxel_id",
    "pre_pt_root_id_partition",
)

Candidate partitions: 199


## Download to Parquet

This cell streams the filtered data to disk. It refuses to overwrite an existing file; choose a new `output_path` or deliberately remove the previous output before rerunning.

In [8]:
if output_path.exists():
    raise FileExistsError(f"Output already exists: {output_path.resolve()}")

start = time.perf_counter()
table_plan.sink_parquet(output_path, engine="streaming")
elapsed = time.perf_counter() - start
print(f"Time taken to write filtered table: {elapsed:.1f} seconds")
print(f"Saved: {output_path.resolve()}")

Time taken to write filtered table: 225.8 seconds
Saved: /workspaces/ConnectsCommonConnectivity/data/EM v1dd 1196/minnie_v1822_proofread_axon_synapses.parquet


## Inspect the download

Count rows and display a small preview without loading the entire file into memory.

In [9]:
table = pl.scan_parquet(output_path)
row_count = table.select(pl.len()).collect().item()
print(f"Number of rows in filtered table: {row_count:,}")
table.head(10).collect()

Number of rows in filtered table: 2,653,659


id,size,pre_pt_root_id,post_pt_root_id,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z
i64,f64,i64,i64,i32,i32,i32
102394584,312.0,864691136040834558,864691135104674916,145336,144930,20737
96547269,5656.0,864691136040834558,864691135426828018,141284,153792,19724
109361181,1660.0,864691136040834558,864691136040834558,150046,147776,18961
303110603,33124.0,864691136040834558,864691135566443843,261665,176117,15683
105924348,864.0,864691136040834558,864691136905290930,145478,169912,18580
118905851,256.0,864691136040834558,864691136965812942,151958,156906,20380
166854992,760.0,864691136040834558,864691135565602839,182624,140492,18321
189418131,1692.0,864691136040834558,864691136298957339,195228,191630,25910
365302775,6048.0,864691136040834558,864691136206839996,300784,164128,15033
